In [1]:
# 1. Install necessary libraries
!pip install dagshub mlflow --quiet

import pandas as pd
import numpy as np
import mlflow.sklearn
import dagshub
import os

# 2. Initialize DagsHub (Replace with your actual token if asked, or it will prompt you)
dagshub.init(repo_owner='ggzob23', repo_name='ML_assignment1', mlflow=True)




     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 76.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 57.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 43.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=9e213e2d-9738-4e05-9839-ceb8e5404164&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e1466773d8db53eb2021ac7dc04e44a91b09b7799d9762024d0129ba47ca9355




Accessing as ggzob23

Initialized MLflow to track repo "ggzob23/ML_assignment1"

Repository ggzob23/ML_assignment1 initialized!

In [2]:
# 3. Load the Test Data from Kaggle's input folder
test_path = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv'
test_df = pd.read_csv(test_path)
test_ids = test_df['Id']


In [3]:
# 4. Cleaning & Encoding Logic (UPDATED to match Training File)
def prepare_test(df):
    temp = df.copy()
    
    cols_to_drop = ['PoolQC', 'MiscFeature', 'Alley', 'Fence']
    temp = temp.drop(columns=[col for col in cols_to_drop if col in temp.columns])
    
    
    num_cols = temp.select_dtypes(include=[np.number]).columns
    temp[num_cols] = temp[num_cols].fillna(temp[num_cols].median())
    
    cat_cols = temp.select_dtypes(include=['object']).columns
    for col in cat_cols:
        temp[col] = temp[col].fillna("None")
        temp[col] = pd.factorize(temp[col])[0]
    return temp

X_test_cleaned = prepare_test(test_df)



In [4]:
# 5. Load the CHAMPION model from DagsHub Registry

model_uri = "models:/HousePriceBestModel/latest" 
loaded_model = mlflow.sklearn.load_model(model_uri)
print(f" Model loaded successfully from: {model_uri}")



 Model loaded successfully from: models:/HousePriceBestModel/latest


In [5]:
# 6. Feature Selection (USE ALL COLUMNS for the Base Linear Model)
X_test_final = X_test_cleaned.copy()

X_test_final = X_test_final[loaded_model.feature_names_in_]

print(f" Data prepared with {X_test_final.shape[1]} features to match the Base Model.")

 Data prepared with 30 features to match the Base Model.


In [6]:
# 7. Generate Predictions
predictions = loaded_model.predict(X_test_final)

# 8. Create Submission File
submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": predictions
})

# Final Safety Check: Ensure no negative prices
submission['SalePrice'] = submission['SalePrice'].clip(lower=0)

submission.to_csv("submission.csv", index=False)
print(" submission.csv created and ready for Kaggle!")

 submission.csv created and ready for Kaggle!
